In [5]:
import pandas as pd

train_df = pd.read_parquet("~/Downloads/qasper-rag/data/train_qasper.parquet") 
test_df = pd.read_parquet("~/Downloads/qasper-rag/data/test_qasper.parquet") 
validation_df = pd.read_parquet("~/Downloads/qasper-rag/data/validation_qasper.parquet")

print(df.shape)
print(df.columns.tolist())
print(df.iloc[0]["title"])
print(df.iloc[0]["full_text"])

(888, 6)
['id', 'title', 'abstract', 'full_text', 'qas', 'figures_and_tables']
Minimally Supervised Learning of Affective Events Using Discourse Relations
{'section_name': array(['Introduction', 'Related Work', 'Proposed Method',
       'Proposed Method ::: Polarity Function',
       'Proposed Method ::: Discourse Relation-Based Event Pairs',
       'Proposed Method ::: Discourse Relation-Based Event Pairs ::: AL (Automatically Labeled Pairs)',
       'Proposed Method ::: Discourse Relation-Based Event Pairs ::: CA (Cause Pairs)',
       'Proposed Method ::: Discourse Relation-Based Event Pairs ::: CO (Concession Pairs)',
       'Proposed Method ::: Loss Functions', 'Experiments',
       'Experiments ::: Dataset',
       'Experiments ::: Dataset ::: AL, CA, and CO',
       'Experiments ::: Dataset ::: ACP (ACP Corpus)',
       'Experiments ::: Model Configurations',
       'Experiments ::: Results and Discussion', 'Conclusion',
       'Acknowledgments',
       'Appendices ::: Seed Lexi

In [7]:
def transform_papers(df):
    records = []

    for _, paper in df.iterrows():
        for section, section_paragraphs in zip(
            paper["full_text"]["section_name"],
            paper["full_text"]["paragraphs"]
        ):
            for paragraph_id, text in enumerate(section_paragraphs):
                text = text.strip()
                if text:
                    records.append({
                        "paper_id": paper["id"],
                        "title": paper["title"],
                        "section": section,
                        "paragraph_id": paragraph_id,
                        "text": text
                    })

    return pd.DataFrame(records, columns=[
        "paper_id", "title", "section", "paragraph_id", "text"
    ])


# Replace train_df, test_df, and validation_df with your existing names.
train_transformed = transform_papers(train_df)
test_transformed = transform_papers(test_df)
validation_transformed = transform_papers(validation_df)

In [11]:
from transformers import AutoTokenizer

# Use this same model when we generate embeddings later.
model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_paragraphs(df):
    result = df.copy()

    encoded = tokenizer(
        result["text"].tolist(),
        add_special_tokens=False,
        truncation=False,
        padding=False,
        return_attention_mask=False,
        return_token_type_ids=False
    )

    result["token_ids"] = encoded["input_ids"]
    result["token_count"] = result["token_ids"].map(len)
    return result

train_transformed = tokenize_paragraphs(train_transformed)
validation_transformed = tokenize_paragraphs(validation_transformed)
test_transformed = tokenize_paragraphs(test_transformed)

print(train_transformed[["text", "token_count"]].head())
print(train_transformed["token_count"].describe())

  Using cached transformers-5.18.0-py3-none-any.whl.metadata (32 kB)
  Using cached regex-2026.9.29-cp311-cp311-win_amd64.whl.metadata (41 kB)
  Using cached tokenizers-0.23.2-cp310-abi3-win_amd64.whl.metadata (10 kB)
  Using cached typer-0.27.2-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached hf_xet-1.6.0-cp38-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached huggingface_hub-1.33.0-py3-none-any.whl.metadata (16 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached annotated_doc-0.0.5-py3-none-any.whl.metadata (6.5 kB)
  Using cached markdown_it_py-4.2.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
Using cached transformers-5.18.0-py3-none-any.whl (12.6 MB)
Using cached hf_xet-1.6.0-cp38-abi3-win_amd64.

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

c:\Users\beepo\Downloadsminiconda3\envs\tuffenv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\beepo\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (888 > 512). Running this sequence through the model will result in indexing errors


                                                text  token_count
0  Affective events BIBREF0 are events that typic...          125
1  Learning affective events is challenging becau...           75
2  In this paper, we propose a simple and effecti...          342
3  We trained the models using a Japanese web cor...           50
4  Learning affective events is closely related t...          144
count    46882.000000
mean       103.818182
std         84.602015
min          1.000000
25%         43.000000
50%         87.000000
75%        143.000000
max       3636.000000
Name: token_count, dtype: float64


In [12]:
def chunk_paragraphs(df, max_tokens=250, overlap=40):
    if not 0 <= overlap < max_tokens:
        raise ValueError("Require 0 <= overlap < max_tokens")

    records = []

    for _, row in df.iterrows():
        token_ids = row["token_ids"]

        for chunk_id, start in enumerate(
            range(0, len(token_ids), max_tokens - overlap)
        ):
            end = min(start + max_tokens, len(token_ids))
            chunk_tokens = token_ids[start:end]

            record = row.to_dict()
            record.update({
                "text": tokenizer.decode(
                    chunk_tokens,
                    skip_special_tokens=True,
                    clean_up_tokenization_spaces=False
                ),
                "token_ids": chunk_tokens,
                "token_count": len(chunk_tokens),
                "chunk_id": chunk_id,
                "token_start": start,
                "token_end": end
            })
            records.append(record)

            # Stop once the entire paragraph has been included.
            if end == len(token_ids):
                break

    return pd.DataFrame(records)

train_chunks = chunk_paragraphs(train_transformed)
validation_chunks = chunk_paragraphs(validation_transformed)
test_chunks = chunk_paragraphs(test_transformed)

for name, chunks in [
    ("train", train_chunks),
    ("validation", validation_chunks),
    ("test", test_chunks)
]:
    print(
        f"{name}: {len(chunks):,} chunks; "
        f"largest = {chunks['token_count'].max()} tokens"
    )

train: 49,569 chunks; largest = 250 tokens
validation: 13,946 chunks; largest = 250 tokens
test: 20,799 chunks; largest = 250 tokens


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(model_name)

print("Model token limit:", model.max_seq_length)

def embed_chunks(chunks):
    return model.encode(
        chunks["text"].tolist(),
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

train_embeddings = embed_chunks(train_chunks)
validation_embeddings = embed_chunks(validation_chunks)
test_embeddings = embed_chunks(test_chunks)

print("Train:", train_embeddings.shape)
print("Validation:", validation_embeddings.shape)
print("Test:", test_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

c:\Users\beepo\Downloadsminiconda3\envs\tuffenv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\beepo\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model token limit: 256


Batches:   0%|          | 0/1550 [00:00<?, ?it/s]

  Using cached sentence_transformers-6.1.0-py3-none-any.whl.metadata (20 kB)
Using cached sentence_transformers-6.1.0-py3-none-any.whl (740 kB)
Note: you may need to restart the kernel to use updated packages.
